<a href="https://colab.research.google.com/github/ayeza527/tmdb_project/blob/main/tmdb_proj_pt2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
os.makedirs("/content/data", exist_ok=True)

SECTION 5:  Data Cleaning and Preprocessing

5A,B

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("/content/data/tmdb_movies.csv")
df.columns = df.columns.str.strip()

df_backup = df.copy()

cols_with_zeros = ["budget", "revenue", "runtime"]

empty_counts = df.isna().sum()

zero_counts = (df[cols_with_zeros] == 0).sum()

summary_table = pd.DataFrame({"empty": empty_counts, "zeros": zero_counts})
summary_table = summary_table.fillna(0).astype(int)

summary_table["total_missing"] = summary_table["empty"] + summary_table["zeros"]
summary_table["percent_missing"] = (summary_table["total_missing"] / len(df) * 100).round(2)

print(summary_table[summary_table["total_missing"] > 0])

                      empty  zeros  total_missing  percent_missing
budget                    0    159            159            14.45
production_companies      1      0              1             0.09
production_countries      2      0              2             0.18
revenue                   0    201            201            18.27
tagline                  43      0             43             3.91
top_cast                  2      0              2             0.18


Before changing anything, I made a backup copy of the data and counted how much is missing in each column. I counted two things: cells that are empty, and zeros in budget, revenue and runtime, because TMDB writes 0 when it does not know these values, even though a movie cannot really have a budget, revenue or runtime of 0. I only checked zeros in these three columns because in other columns a zero can be a real value. The table shows that budget has 159 missing values (14.45%) and revenue has 201 (18.27%), while runtime has none. The text columns have only a few empty cells: tagline 43 (3.91%), production_countries 2, top_cast 2 and production_companies 1. This tells me the zeros in budget and revenue are the main missing-data problem, and that 899 of the 1,100 movies have a real revenue value, which is enough to use revenue as the target.

In [ ]:
print("Exact duplicate rows:", df.duplicated().sum())
print("Repeated movie ids:", df["id"].duplicated().sum())

rows_before = len(df)
df = df.drop_duplicates()
print("Removed:", rows_before - len(df), " rows left:", len(df))

Exact duplicate rows: 0
Repeated movie ids: 0
Removed: 0  rows left: 1100


I checked whether any movie appears more than once, both as an exact copy of a whole row and as a repeated movie id, and then dropped any exact duplicate rows. I did this because duplicate movies would be counted twice and could give the models an unfair advantage when the same movie lands in both the training and test sets. The output shows 0 duplicate rows and 0 repeated ids, so no rows were removed and 1,100 rows are left. This is what I expected, because I already removed duplicate ids when I collected the movie ids, so the data had no duplicates to remove.

In [ ]:
df[cols_with_zeros] = df[cols_with_zeros].replace(0, np.nan)

print(df[cols_with_zeros].isna().sum())
print("Rows with a real revenue:", df["revenue"].notna().sum())

budget     159
revenue    201
runtime      0
dtype: int64
Rows with a real revenue: 899


I replaced the zeros in budget, revenue and runtime with proper missing values (NaN). I did this because TMDB writes 0 when it does not know a value, but a movie cannot really have a budget, revenue or runtime of 0, so leaving the zeros would make the averages, charts and models wrong. The output shows that budget now has 159 missing values, revenue has 201 and runtime has 0, which matches the zero counts I measured before, and that 899 movies have a real revenue value. Since I plan to use revenue as the target, the movies with a missing revenue will be left out of model training and testing, and the rest will be used.

In [ ]:

df_copy = df.copy()

cols = [
    "budget",
    "runtime",
    "popularity",
    "vote_average",
    "vote_count",
    "cast_size"
]

rng = np.random.default_rng(900)

for col in cols:
    valid_rows = df_copy.index[df_copy[col].notna()]
    missing_count = round(0.05 * len(valid_rows))

    missing_rows = rng.choice(
        valid_rows.to_numpy(),
        size=missing_count,
        replace=False
    )

    df_copy.loc[missing_rows, col] = np.nan

comparison = pd.DataFrame({
    "Before": df[cols].isna().sum(),
    "After": df_copy[cols].isna().sum()
})

print(comparison)

print("\nRevenue missing values:", df_copy["revenue"].isna().sum())

df_copy.to_csv("/content/data/tmdb_movies_with_missing.csv", index=False)


              Before  After
budget           159    206
runtime            0     55
popularity         0     55
vote_average       0     55
vote_count         0     55
cast_size          0     55

Revenue missing values: 201


To evaluate how well our data handles missing values, we created a separate copy of the original dataset and introduced approximately 5% artificial missing values in selected input features. Randomly selected non-missing values in each selected column were replaced with NaN, while the original dataset was kept unchanged. We then compared the number of missing values before and after this process to observe the increase in missing data. We also verified that the target variable (revenue) remained unaffected. This modified dataset will be used to evaluate different missing value handling and imputation techniques.

In [ ]:
df_copy.dtypes

,0
id,int64
title,object
release_date,object
budget,float64
revenue,float64
runtime,float64
popularity,float64
vote_average,float64
vote_count,float64
original_language,object


In [ ]:
df_model = df_copy.dropna(subset=["revenue"]).copy()

df_model["release_date"] = pd.to_datetime(df_model["release_date"])
df_model["release_year"] = df_model["release_date"].dt.year
df_model["release_month"] = df_model["release_date"].dt.month
df_model["n_genres"] = df_model["genres"].fillna("").apply(lambda g: len(g.split("|")) if g else 0)

print(df_model.shape)

from sklearn.model_selection import train_test_split

SEED = 900
X = df_model.drop(columns=["revenue"])
y = df_model["revenue"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
X_train_raw = X_train.copy()
X_test_raw = X_test.copy()
print(X_train.shape, X_test.shape)

(899, 26)
(719, 25) (180, 25)


I kept only the movies with a real revenue value (899 of 1,100) and split them randomly into 80% training data and 20% test data, using random_state=900, the team’s seed, so that the split is the same every time the notebook is run. I chose a simple random split because each row is a different movie (I checked that there are no repeated ids, so the same movie cannot end up in both sets), and because I am predicting revenue from the features of each movie, not forecasting over time, so the order of the movies does not matter. The dataset is also small, and a random split mixes movies from all years (2016 to 2026) in both sets, while a split by year would leave the test set with only a few specific years, and the score would depend on which years were chosen. The output shows 719 training rows and 180 test rows. I did the split before any imputation, encoding or scaling, so these steps can be fitted on the training data only, and the test set stays untouched until the final evaluation. A limitation is that a random split does not show how well the model predicts movies released in the future, which a split by release date would test better, and because revenue is very skewed, a random split could by chance put more blockbusters in one set than the other, which is why I also use cross-validation to check that the results are stable.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
X_train = X_train_raw.copy()
X_test = X_test_raw.copy()
leak_cols = ["vote_count", "vote_average", "popularity"]
not_features = ["id", "title", "overview", "tagline", "collected_at", "release_date"]
X_train = X_train.drop(columns=leak_cols + not_features, errors="ignore")
X_test = X_test.drop(columns=leak_cols + not_features, errors="ignore")

def count_items(text):
    return len(text.split("|")) if isinstance(text, str) and text else 0

X_train["n_companies"] = X_train["production_companies"].apply(count_items)
X_test["n_companies"] = X_test["production_companies"].apply(count_items)

G_train = X_train["genres"].fillna("").str.get_dummies(sep="|").add_prefix("g_")
G_test = X_test["genres"].fillna("").str.get_dummies(sep="|").add_prefix("g_")
G_test = G_test.reindex(columns=G_train.columns, fill_value=0)

C_train = X_train["production_countries"].fillna("").str.get_dummies(sep="|").add_prefix("c_")
C_train = C_train.loc[:, C_train.sum() >= 20]
C_test = X_test["production_countries"].fillna("").str.get_dummies(sep="|").add_prefix("c_")
C_test = C_test.reindex(columns=C_train.columns, fill_value=0)

X_train = X_train.join(G_train).join(C_train)
X_test = X_test.join(G_test).join(C_test)

num_cols = ["budget", "runtime", "cast_size", "spoken_languages",
            "release_year", "release_month", "n_genres", "n_companies"]
cat_cols = ["original_language"]
bool_cols = ["adult", "in_collection"]
multi_cols = list(G_train.columns) + list(C_train.columns)

X_train[bool_cols] = X_train[bool_cols].astype(int)
X_test[bool_cols] = X_test[bool_cols].astype(int)

num_steps = Pipeline([("fill", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())])
cat_steps = Pipeline([("fill", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))])

prep = ColumnTransformer([
    ("num", num_steps, num_cols),
    ("cat", cat_steps, cat_cols),
    ("keep", "passthrough", bool_cols + multi_cols),
], sparse_threshold=0)

print("Missing before:")
print(X_train[num_cols + cat_cols].isna().sum())

X_train_ready = prep.fit_transform(X_train)
X_test_ready = prep.transform(X_test)

print("Missing after:", np.isnan(X_train_ready).sum())
print(X_train_ready.shape, X_test_ready.shape)

Missing before:
budget               62
runtime              30
cast_size            40
spoken_languages      0
release_year          0
release_month         0
n_genres              0
n_companies           0
original_language     0
dtype: int64
Missing after: 0
(719, 48) (180, 48)


In this step I prepared the input data for the models, and I fitted every step on the training data only and then applied it to the test data without refitting, so that nothing from the test set leaks into training. First I left out the columns that should not be inputs: vote_count, vote_average and popularity because they only exist after a movie is released and would leak the revenue, id and title because they are only labels, release_date because I already took the year and month from it, collected_at because it only records when I collected the data, and overview and tagline because they are free text that needs special text methods. I also left out director and top_cast, because they have hundreds of different names for only about 719 training movies, which would create a huge number of almost empty columns and make the model memorise instead of learn. Instead of dropping the production columns completely, I kept some of their information in a simpler form: n_companies counts how many companies made each movie, and production_countries is split into 0/1 columns, keeping only countries that appear in at least 20 training movies so that rare countries do not create useless columns. The genres are handled the same way: because a movie can have several genres in one cell separated by “|”, I split them into one 0/1 column per genre (for example g_Action), and I forced the test set to have exactly the same columns as the training set. The Boolean columns adult and in_collection were turned into 1 and 0. For the number columns, missing values were filled with the median, because budget has a few huge values that would pull the average up, and then they were scaled with StandardScaler so that budget, which is in millions, does not dominate runtime, which is around 100 minutes, just because its numbers are bigger. For the language column, missing values were filled with the most common language, because a median cannot be taken from text, and the column was one-hot encoded, with unknown languages in the test set ignored so the code does not crash. Before treatment, the training inputs had about 5% missing values in budget (), runtime () and cast_size (), the columns where I introduced the artificial missing values (budget also has its real missing values from TMDB’s zeros), and after treatment the number of missing values is , so the comparison shows that the imputation worked. The final shapes are ___ columns for the training data ( rows) and ___ columns for the test data ( rows), the same number of columns in both. A limitation is that I left out the director, the cast and the individual production companies, so the model cannot learn the effect of a famous director or a big studio, which probably does affect revenue, and I would need more movies or a more advanced technique such as target encoding to use them properly.